In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from config.settings import *
from config.feature_registry import (
    OPEN_METEO_HOURLY_PARAMS, NASA_POWER_PARAMS,
    NASA_COLUMN_RENAME, FEATURE_METADATA,
    get_tier1_features, get_tier2_features, get_domain_features,
)

# Plot settings
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 11
sns.set_theme(style="whitegrid")

print("✅ Setup complete")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  LOAD COMBINED DATASETS
# ══════════════════════════════════════════════════════════════

# Open-Meteo
om_df = pd.read_csv(INTERIM_DIR / "open_meteo_combined.csv")
om_df["date"] = pd.to_datetime(om_df["date"])
print(f"🌤️  Open-Meteo: {om_df.shape}")
print(f"   Date range: {om_df['date'].min()} → {om_df['date'].max()}")
print(f"   Points: {om_df[['latitude','longitude']].drop_duplicates().shape[0]}")

# NASA POWER
nasa_df = pd.read_csv(INTERIM_DIR / "nasa_power_combined.csv")
nasa_df["date"] = pd.to_datetime(nasa_df["date"])
print(f"\n🛰️  NASA POWER: {nasa_df.shape}")
print(f"   Date range: {nasa_df['date'].min()} → {nasa_df['date'].max()}")
print(f"   Points: {nasa_df[['latitude','longitude']].drop_duplicates().shape[0]}")

# Grid points
grid_df = pd.read_csv(INTERIM_DIR / "grid_points_india.csv")
print(f"\n📍 Grid points: {len(grid_df)}")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  BASIC STATISTICS — OPEN METEO
# ══════════════════════════════════════════════════════════════

print("=" * 70)
print("  OPEN-METEO DATA OVERVIEW")
print("=" * 70)

print(f"\n📊 Shape: {om_df.shape}")
print(f"📅 Date range: {om_df['date'].min().date()} → {om_df['date'].max().date()}")
print(f"📍 Unique locations: {om_df[['latitude','longitude']].drop_duplicates().shape[0]}")

print(f"\n📋 Column Types:")
print(f"   Numeric: {len(om_df.select_dtypes(include=[np.number]).columns)}")
print(f"   Non-numeric: {len(om_df.select_dtypes(exclude=[np.number]).columns)}")

print(f"\n📊 Missing Values:")
missing = om_df.isnull().sum()
missing_pct = (missing / len(om_df) * 100).round(2)
missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct,
}).sort_values("missing_pct", ascending=False)

print(missing_report[missing_report["missing_count"] > 0].to_string())

if missing_report["missing_count"].sum() == 0:
    print("   ✅ No missing values!")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  DESCRIPTIVE STATISTICS
# ══════════════════════════════════════════════════════════════

# Key weather columns
key_cols = [
    "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
    "precipitation_sum", "relative_humidity_2m_mean",
    "wind_speed_10m_mean", "cloud_cover_mean",
    "pressure_msl_mean", "shortwave_radiation_sum",
]
available_key = [c for c in key_cols if c in om_df.columns]

print("📊 Key Variable Statistics:")
print("=" * 80)
stats = om_df[available_key].describe().round(2)
print(stats.to_string())

# Additional percentiles
print("\n📊 Additional Percentiles:")
for col in available_key:
    p1 = om_df[col].quantile(0.01)
    p99 = om_df[col].quantile(0.99)
    iqr = om_df[col].quantile(0.75) - om_df[col].quantile(0.25)
    print(f"   {col:<35s} P1={p1:>8.2f}  P99={p99:>8.2f}  IQR={iqr:>8.2f}")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  DISTRIBUTION OF KEY VARIABLES
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for idx, col in enumerate(available_key[:9]):
    ax = axes[idx]
    data = om_df[col].dropna()

    ax.hist(data, bins=80, color="steelblue", alpha=0.7, edgecolor="white")
    ax.axvline(data.mean(), color="red", linestyle="--", linewidth=1.5, label=f"Mean: {data.mean():.1f}")
    ax.axvline(data.median(), color="green", linestyle="--", linewidth=1.5, label=f"Median: {data.median():.1f}")

    ax.set_title(col.replace("_", " ").title(), fontsize=11, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("Distribution of Key Weather Variables (All Points, All Days)", 
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "eda_distributions.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CORRELATION MATRIX
# ══════════════════════════════════════════════════════════════

# Use key columns for readable heatmap
corr_cols = [c for c in available_key if c in om_df.columns]
corr_matrix = om_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"shrink": 0.8},
)

# Clean labels
labels = [c.replace("_", "\n").replace("2m\nmean", "2m") for c in corr_cols]
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(labels, rotation=0, fontsize=9)

ax.set_title("Feature Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "eda_correlation_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()

# Print strong correlations
print("🔗 Strong Correlations (|r| > 0.7):")
for i in range(len(corr_cols)):
    for j in range(i + 1, len(corr_cols)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.7:
            print(f"   {corr_cols[i]:<35s} ↔ {corr_cols[j]:<35s} r={r:.3f}")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  TEMPORAL PATTERNS — MONTHLY AVERAGES ACROSS ALL INDIA
# ══════════════════════════════════════════════════════════════

om_df["month"] = om_df["date"].dt.month
om_df["year"] = om_df["date"].dt.year

monthly_avg = om_df.groupby("month")[available_key].mean()

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

colors = ["#e74c3c", "#e74c3c", "#e74c3c",  # temp
          "#3498db",  # precip
          "#2ecc71",  # humidity
          "#9b59b6",  # wind
          "#95a5a6",  # cloud
          "#e67e22",  # pressure
          "#f1c40f"]  # radiation

for idx, col in enumerate(available_key[:9]):
    ax = axes[idx]
    values = monthly_avg[col].values

    ax.bar(range(1, 13), values, color=colors[idx], alpha=0.8, edgecolor="white")
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(month_names, fontsize=8)
    ax.set_title(col.replace("_", " ").title(), fontsize=11, fontweight="bold")
    ax.grid(True, alpha=0.3, axis="y")

    # Highlight monsoon months
    for m in [6, 7, 8, 9]:
        ax.get_children()[m - 1].set_edgecolor("darkblue")
        ax.get_children()[m - 1].set_linewidth(2)

plt.suptitle("Monthly Average Weather Patterns — India\n(Blue borders = Monsoon months)", 
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "eda_monthly_patterns.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
#  SPATIAL DISTRIBUTION — AVG VALUES PER GRID POINT
# ══════════════════════════════════════════════════════════════

point_avg = om_df.groupby(["latitude", "longitude"])[available_key].mean().reset_index()

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

plot_features = [
    ("temperature_2m_mean", "Mean Temperature (°C)", "RdYlBu_r"),
    ("precipitation_sum", "Mean Daily Precipitation (mm)", "YlGnBu"),
    ("relative_humidity_2m_mean", "Mean Humidity (%)", "YlGnBu"),
    ("wind_speed_10m_mean", "Mean Wind Speed (km/h)", "YlOrRd"),
    ("cloud_cover_mean", "Mean Cloud Cover (%)", "Greys"),
    ("shortwave_radiation_sum", "Mean Solar Radiation (W/m²)", "YlOrRd"),
]

for idx, (col, title, cmap) in enumerate(plot_features):
    if col not in point_avg.columns:
        continue
    ax = axes[idx]

    scatter = ax.scatter(
        point_avg["longitude"],
        point_avg["latitude"],
        c=point_avg[col],
        cmap=cmap,
        s=20,
        alpha=0.8,
        edgecolors="none",
    )

    plt.colorbar(scatter, ax=ax, shrink=0.8)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_xlim(67, 98)
    ax.set_ylim(7, 38)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.2)

plt.suptitle("Spatial Distribution of Weather Variables — India", 
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "eda_spatial_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
#  REGIONAL COMPARISON — NORTH vs SOUTH vs EAST vs WEST vs CENTRAL
# ══════════════════════════════════════════════════════════════

def assign_region(row):
    lat, lon = row["latitude"], row["longitude"]
    if lat > 28:
        return "North"
    elif lat < 15:
        return "South"
    elif lon > 85:
        return "East"
    elif lon < 73:
        return "West"
    else:
        return "Central"

om_df["region"] = om_df.apply(assign_region, axis=1)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

region_order = ["North", "South", "East", "West", "Central"]
palette = {"North": "#3498db", "South": "#e74c3c", "East": "#2ecc71", 
           "West": "#f39c12", "Central": "#9b59b6"}

for idx, col in enumerate(available_key[:6]):
    ax = axes[idx]
    
    # Sample for speed (boxplots with millions of points are slow)
    sample = om_df.groupby("region").apply(
        lambda x: x.sample(min(5000, len(x)), random_state=42)
    ).reset_index(drop=True)
    
    sns.boxplot(
        data=sample,
        x="region",
        y=col,
        order=region_order,
        palette=palette,
        ax=ax,
        fliersize=1,
    )
    ax.set_title(col.replace("_", " ").title(), fontsize=11, fontweight="bold")
    ax.set_xlabel("")
    ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("Weather Variable Distribution by Region", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "eda_regional_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

# Print regional statistics
print("📊 Regional Statistics (Means):")
regional_stats = om_df.groupby("region")[available_key].mean().round(2)
print(regional_stats[available_key[:6]].to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════
#  MONSOON vs NON-MONSOON ANALYSIS
# ══════════════════════════════════════════════════════════════

SEASONS = {
    "Winter": [12, 1, 2],
    "Pre-Monsoon": [3, 4, 5],
    "Monsoon": [6, 7, 8, 9],
    "Post-Monsoon": [10, 11],
}

month_to_season = {}
for season, months in SEASONS.items():
    for m in months:
        month_to_season[m] = season

om_df["season"] = om_df["month"].map(month_to_season)

# Seasonal averages
seasonal_avg = om_df.groupby("season")[available_key].mean()
season_order = ["Winter", "Pre-Monsoon", "Monsoon", "Post-Monsoon"]
seasonal_avg = seasonal_avg.reindex(season_order)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

season_colors = {
    "Winter": "#3498db",
    "Pre-Monsoon": "#f39c12",
    "Monsoon": "#27ae60",
    "Post-Monsoon": "#e74c3c",
}

for idx, col in enumerate(available_key[:6]):
    ax = axes[idx]
    values = [seasonal_avg.loc[s, col] for s in season_order]
    colors = [season_colors[s] for s in season_order]

    bars = ax.bar(season_order, values, color=colors, alpha=0.8, edgecolor="white")
    ax.set_title(col.replace("_", " ").title(), fontsize=11, fontweight="bold")
    ax.grid(True, alpha=0.3, axis="y")
    ax.set_xticklabels(season_order, rotation=15, fontsize=9)

    # Add value labels
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{val:.1f}", ha="center", va="bottom", fontsize=9)

plt.suptitle("Seasonal Weather Patterns — India", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "eda_seasonal_patterns.png"), dpi=150, bbox_inches="tight")
plt.show()

print("📊 Seasonal Statistics:")
print(seasonal_avg[available_key[:6]].round(2).to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════
#  YEARLY TRENDS — IS CLIMATE CHANGING?
# ══════════════════════════════════════════════════════════════

yearly_avg = om_df.groupby("year")[available_key].mean()

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

trend_features = [
    ("temperature_2m_mean", "Mean Temperature", "°C"),
    ("precipitation_sum", "Daily Precipitation", "mm"),
    ("relative_humidity_2m_mean", "Humidity", "%"),
    ("wind_speed_10m_mean", "Wind Speed", "km/h"),
    ("cloud_cover_mean", "Cloud Cover", "%"),
    ("shortwave_radiation_sum", "Solar Radiation", "W/m²"),
]

for idx, (col, title, unit) in enumerate(trend_features):
    if col not in yearly_avg.columns:
        continue
    ax = axes[idx]

    years = yearly_avg.index
    values = yearly_avg[col].values

    ax.plot(years, values, "o-", color="steelblue", linewidth=2, markersize=4)

    # Add trend line
    z = np.polyfit(years, values, 1)
    p = np.poly1d(z)
    ax.plot(years, p(years), "--", color="red", linewidth=2,
            label=f"Trend: {z[0]:+.4f} {unit}/year")

    ax.set_title(f"{title} ({unit})", fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Year")

plt.suptitle("Yearly Trends — Climate Change Signals in India", 
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "eda_yearly_trends.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
#  HOURLY DERIVED FEATURES ANALYSIS
# ══════════════════════════════════════════════════════════════

hourly_features = [
    "frost_hours", "hot_hours", "comfortable_hours",
    "precipitation_hours", "dry_hours",
    "calm_hours", "strong_wind_hours",
    "high_humidity_hours", "low_cloud_hours", "overcast_hours",
]
available_hourly = [c for c in hourly_features if c in om_df.columns]

if available_hourly:
    # Average hourly features per point
    point_hourly = om_df.groupby(["latitude", "longitude"])[available_hourly].mean().reset_index()

    fig, axes = plt.subplots(2, 5, figsize=(24, 9))
    axes = axes.flatten()

    for idx, col in enumerate(available_hourly[:10]):
        ax = axes[idx]
        scatter = ax.scatter(
            point_hourly["longitude"],
            point_hourly["latitude"],
            c=point_hourly[col],
            cmap="YlOrRd",
            s=15,
            alpha=0.8,
        )
        plt.colorbar(scatter, ax=ax, shrink=0.7)
        ax.set_title(col.replace("_", " ").title(), fontsize=9, fontweight="bold")
        ax.set_xlim(67, 98)
        ax.set_ylim(7, 38)
        ax.set_aspect("equal")
        ax.tick_params(labelsize=7)

    plt.suptitle("Hourly Derived Features — Spatial Distribution", 
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / "eda_hourly_derived.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("⚠️  No hourly derived features found")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  NASA POWER DATA EXPLORATION
# ══════════════════════════════════════════════════════════════

print("🛰️  NASA POWER Data Overview:")
print(f"Shape: {nasa_df.shape}")
print(f"\nColumn Stats:")

nasa_cols = [c for c in nasa_df.columns if c not in ["date", "latitude", "longitude"]]

for col in nasa_cols:
    non_null = nasa_df[col].notna().sum()
    pct = non_null / len(nasa_df) * 100
    mean = nasa_df[col].mean()
    print(f"   {col:<25s} {pct:5.1f}% data | mean={mean:.2f}")

# Key NASA features spatial plot
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

nasa_features = [
    ("ALLSKY_SFC_SW_DWN", "Solar Irradiance (All Sky)", "YlOrRd"),
    ("CLRSKY_SFC_SW_DWN", "Solar Irradiance (Clear Sky)", "YlOrRd"),
    ("TS", "Earth Skin Temperature (°C)", "RdYlBu_r"),
    ("QV2M", "Specific Humidity (g/kg)", "YlGnBu"),
    ("CLOUD_AMT", "Cloud Amount (%)", "Greys"),
    ("ALLSKY_SFC_UV_INDEX", "UV Index", "Purples"),
]

nasa_point_avg = nasa_df.groupby(["latitude", "longitude"])[nasa_cols].mean().reset_index()

for idx, (col, title, cmap) in enumerate(nasa_features):
    if col not in nasa_point_avg.columns:
        continue
    ax = axes[idx]

    scatter = ax.scatter(
        nasa_point_avg["longitude"],
        nasa_point_avg["latitude"],
        c=nasa_point_avg[col],
        cmap=cmap,
        s=15,
        alpha=0.8,
    )
    plt.colorbar(scatter, ax=ax, shrink=0.8)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlim(67, 98)
    ax.set_ylim(7, 38)
    ax.set_aspect("equal")

plt.suptitle("NASA POWER Data — Spatial Distribution", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "eda_nasa_power_spatial.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CROSS-VALIDATION: OPEN-METEO vs NASA POWER
# ══════════════════════════════════════════════════════════════

# Merge on rounded coordinates and date for comparison
om_sample = om_df[["date", "latitude", "longitude", "temperature_2m_mean",
                     "precipitation_sum", "wind_speed_10m_mean"]].copy()
om_sample["lat_r"] = om_sample["latitude"].round(1)
om_sample["lon_r"] = om_sample["longitude"].round(1)

nasa_sample = nasa_df[["date", "latitude", "longitude", "T2M", "PRECTOTCORR", "WS10M"]].copy()
nasa_sample["lat_r"] = nasa_sample["latitude"].round(1)
nasa_sample["lon_r"] = nasa_sample["longitude"].round(1)

merged_check = pd.merge(
    om_sample, nasa_sample,
    on=["lat_r", "lon_r", "date"],
    how="inner",
    suffixes=("_om", "_nasa"),
)

print(f"📊 Cross-validation pairs: {len(merged_check):,}")

if len(merged_check) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    comparisons = [
        ("temperature_2m_mean", "T2M", "Temperature (°C)"),
        ("precipitation_sum", "PRECTOTCORR", "Precipitation (mm)"),
        ("wind_speed_10m_mean", "WS10M", "Wind Speed"),
    ]

    for ax, (om_col, nasa_col, title) in zip(axes, comparisons):
        # Sample for speed
        sample = merged_check.sample(min(10000, len(merged_check)), random_state=42)

        ax.scatter(sample[om_col], sample[nasa_col], alpha=0.1, s=1, color="steelblue")

        # Perfect agreement line
        lims = [
            min(sample[om_col].min(), sample[nasa_col].min()),
            max(sample[om_col].max(), sample[nasa_col].max()),
        ]
        ax.plot(lims, lims, "r--", linewidth=2, label="Perfect agreement")

        # Correlation
        corr = sample[om_col].corr(sample[nasa_col])
        ax.set_title(f"{title}\nr = {corr:.3f}", fontsize=11, fontweight="bold")
        ax.set_xlabel("Open-Meteo")
        ax.set_ylabel("NASA POWER")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.suptitle("Cross-Validation: Open-Meteo vs NASA POWER", 
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / "eda_cross_validation.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("⚠️  No matching records found for cross-validation")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  EDA SUMMARY
# ══════════════════════════════════════════════════════════════

print("=" * 70)
print("  📋 EDA SUMMARY")
print("=" * 70)
print(f"""
  📊 Open-Meteo Dataset:
     Rows:           {len(om_df):,}
     Columns:        {len(om_df.columns)}
     Grid points:    {om_df[['latitude','longitude']].drop_duplicates().shape[0]}
     Date range:     {om_df['date'].min().date()} → {om_df['date'].max().date()}
     Missing values: {om_df.isnull().sum().sum():,}
     
  🛰️  NASA POWER Dataset:
     Rows:           {len(nasa_df):,}
     Columns:        {len(nasa_df.columns)}
     Grid points:    {nasa_df[['latitude','longitude']].drop_duplicates().shape[0]}
     
  🔑 Key Findings:
     1. Temperature shows clear latitude gradient (North=cool, South=hot)
     2. Precipitation dominated by monsoon (Jun-Sep)
     3. Strong correlation between temperature and radiation
     4. Regional diversity makes India ideal for clustering
     5. Snow depth only relevant for northern high-altitude points
     6. Hourly derived features reveal day-night patterns
     
  📁 Plots saved to: {FIGURES_DIR}
""")
print("=" * 70)